# YOLO Segmentation

객체 탐지가 이미지 안의 객체 위치를 사각형 박스로 찾는 작업이라면, segmentation은 객체가 차지하는 영역을 픽셀 단위에 가깝게 분리하는 작업이다.

- 이미지 분류: 이미지 전체가 무엇인지 예측한다.
- 객체 탐지: 이미지 안의 객체 위치와 클래스를 예측한다.
- 세그멘테이션: 객체가 차지하는 실제 영역을 마스크로 예측한다.

YOLO의 segmentation 모델은 객체의 class, confidence, bounding box뿐 아니라 mask 정보까지 함께 반환한다.

## Object Detection과 Segmentation의 차이

Object Detection은 객체를 사각형 박스로 감싸고 다음과 같은 정보를 출력한다.

- class: 객체의 종류
- confidence: 모델이 해당 객체라고 판단한 확신도
- bounding box: 객체를 둘러싸는 사각형 영역

반면 Segmentation은 객체의 영역을 더 세밀하게 표현한다.

사각형 박스는 객체가 아닌 배경까지 함께 포함할 수 있지만, segmentation mask는 객체가 실제로 차지하는 영역을 더 구체적으로 나타낸다.

따라서 segmentation은 의료 영상, 자율주행, 배경 제거, 제품 결함 검사, 객체 영역 분석처럼 객체의 정확한 형태가 중요한 작업에서 많이 사용된다.

## Semantic Segmentation과 Instance Segmentation

Segmentation은 크게 semantic segmentation과 instance segmentation으로 나누어 볼 수 있다.

Semantic Segmentation은 같은 클래스에 속한 픽셀을 하나의 의미 영역으로 분류한다. 예를 들어 이미지 안의 모든 사람 영역을 `person`으로 표시한다.

Instance Segmentation은 같은 클래스에 속한 객체라도 각각의 객체를 구분한다. 예를 들어 사람 1, 사람 2, 사람 3을 서로 다른 객체 인스턴스로 분리한다.

Ultralytics YOLO의 `-seg` 모델은 일반적으로 instance segmentation 모델로 이해하면 된다. 즉, 객체를 탐지하고 각 객체별 mask를 함께 예측한다.

In [2]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, display
from ultralytics import YOLO

## 사전학습 YOLO Segmentation 모델 로드

https://docs.ultralytics.com/ko/tasks/segment

Ultralytics YOLO는 detection, segmentation, classification 등 여러 vision task를 지원한다.

Segmentation 모델은 보통 모델 파일명에 `-seg`가 붙는다.

현재 공식 문서는 YOLO26 계열 예제를 기준으로 `yolo26n-seg.pt`를 사용한다. 여기서 `n`은 nano 모델을 의미하며, 크기가 작아 실습용으로 적합하다.

단, 설치된 ultralytics 버전이나 실행 시점에 따라 제공되는 모델명이 다를 수 있다. 만약 `yolo26n-seg.pt` 다운로드가 실패하면 같은 코드 구조에서 `yolo11n-seg.pt` 또는 `yolov8n-seg.pt`로 바꿔 실행하면 된다.

사전학습 YOLO Segmentation 모델도 보통 COCO 데이터셋 기준으로 학습되어 있다. 따라서 사람, 자동차, 버스, 강아지, 고양이 등 COCO에 포함된 일반 객체에 대해 segmentation 결과를 확인할 수 있다.